# production health objective loss v1

验证 production health 从 production manifest 声明的 paper execution evidence 中读取 `objective.loss.v1` gate，并将其传递到 dashboard view/page。

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'production_health_objective_loss_v1'
CARD.mkdir(parents=True, exist_ok=True)
print(f'ROOT={ROOT}')
print(f'CARD={CARD}')

ROOT=/Users/rziiiii/dev/workspace
CARD=/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1


In [2]:
from src.core import (
    DatasetSnapshot,
    build_architecture_extension_decision,
    publish_production_manifest,
    write_architecture_extension_decision,
)
from src.dashboard import build_dashboard_view_model, render_dashboard_html
from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

In [3]:
def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def publish_strategy(strategy_id: str, *, objective_metadata: dict) -> Path:
    strategy_dir = CARD / 'production_cases' / strategy_id
    write_json(strategy_dir / 'production_summary.json', {'status': 'paper_replay_completed', 'strategy_id': strategy_id})
    write_json(strategy_dir / 'dashboard_export.json', {
        'schema_version': 'dashboard.model_matrix.v1',
        'manifest_source': {
            'kind': 'analysis_manifest',
            'path': 'matrix_card/analysis_manifest.json',
            'manifest_hash': 'a' * 64,
            'producer': 'src.analysis.exports',
        },
        'top_models': [{'rank': 1, 'model': 'ridge_mse__all_features', 'avg_position': 0.5}],
    })
    write_json(strategy_dir / 'matrix_card' / 'analysis_manifest.json', {'schema_version': 'artifact.v1'})
    write_json(strategy_dir / 'data' / 'data_registry.json', {'schema_version': 'data.registry.v1'})
    intent = {
        'schema_version': 'api.execution_intent.v1',
        'instrument': 'SOXL',
        'target_position': 0.5 if objective_metadata['objective_loss_status'] != 'block' else 0.1,
        'source_model': 'ridge_mse__all_features',
        'source_artifact': strategy_id,
        'mode': 'paper',
        'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
        'metadata': objective_metadata,
    }
    write_json(strategy_dir / 'paper_execution_result.json', {
        'schema_version': 'api.paper_execution_result.v1',
        'status': 'paper_executed',
        'broker_name': 'paper',
        'intent': intent,
        'snapshot': {'cash': 500.0, 'shares': 41, 'price': 12.0},
    })
    write_json(strategy_dir / 'intent_ledger.json', {
        'schema_version': 'api.intent_ledger.v1',
        'intent': intent,
        'execution_result': {'status': 'paper_executed'},
    })
    strategy_notebook = {
        'cells': [
            {
                'cell_type': 'markdown',
                'metadata': {},
                'source': [
                    f'# {strategy_id}\n',
                    '\n',
                    '## 中文结论\n',
                    '\n',
                    '- 做了什么：作为 production health objective loss 实验的最小 production notebook 证据。\n',
                    '- 核心指标：objective loss 状态由 paper execution intent metadata 提供。\n',
                    '- Regime 差异：本占位 production case 不做 regime 对比。\n',
                    '- 是否晋升：否，仅用于 production health 框架验证。\n',
                    '- 风险：不代表真实策略晋升或真实 broker 执行。\n',
                    '- 下一步：由主研究 notebook 汇总 review/block 对比。\n',
                ],
            }
        ],
        'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}},
        'nbformat': 4,
        'nbformat_minor': 5,
    }
    write_json(strategy_dir / f'{strategy_id}.ipynb', strategy_notebook)
    dataset = DatasetSnapshot(
        name=f'{strategy_id}_soxl_daily',
        universe=('SOXL',),
        start='2026-01-02',
        end='2026-06-09',
        calendar_path='data/qlib/soxl/calendars/day.txt',
        data_paths=('data/data_registry.json',),
    )
    publish_production_manifest(
        strategy_id=strategy_id,
        output_dir=strategy_dir,
        registry_path=CARD / 'production_registry.json',
        notebook_path=f'{strategy_id}.ipynb',
        dataset=dataset,
        paths={
            'summary': 'production_summary.json',
            'dashboard_export': 'dashboard_export.json',
            'intent_ledger': 'intent_ledger.json',
            'paper_execution_result': 'paper_execution_result.json',
            'data_registry': 'data/data_registry.json',
            'analysis_manifest': 'matrix_card/analysis_manifest.json',
        },
        metadata={
            'status': 'paper_replay_completed',
            'mode': 'paper',
            'live_broker_enabled': False,
            'paper_execution_pipeline_schema_version': 'schedule.paper_execution.v1',
            'paper_execution_job_status': 'success',
        },
        published_at='2026-06-19T00:00:00Z',
    )
    return strategy_dir

In [4]:
review_strategy_dir = publish_strategy(
    'paper_review_objective_loss_strategy',
    objective_metadata={
        'objective_loss_schema_version': 'objective.loss.v1',
        'objective_loss_status': 'review',
        'objective_loss_reward_surface': 'retired',
        'objective_loss_optimization_direction': 'minimize',
        'objective_loss_top_component': 'ranking_loss',
        'objective_loss_top_value': 1.0,
    },
)
blocked_strategy_dir = publish_strategy(
    'paper_blocked_objective_loss_strategy',
    objective_metadata={
        'objective_loss_schema_version': 'objective.loss.v1',
        'objective_loss_status': 'block',
        'objective_loss_reward_surface': 'active_rl_reward',
        'objective_loss_optimization_direction': 'minimize',
        'objective_loss_top_component': 'ic_loss',
        'objective_loss_top_value': 0.9774,
    },
)
review_strategy_dir, blocked_strategy_dir

(PosixPath('/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/production_cases/paper_review_objective_loss_strategy'),
 PosixPath('/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/production_cases/paper_blocked_objective_loss_strategy'))

In [5]:
events: list[str] = []
review_config = ProductionHealthCheckConfig(
    name='production_health_review_objective_loss',
    registry_path=CARD / 'production_registry.json',
    strategy_id='paper_review_objective_loss_strategy',
    output_path=CARD / 'production_health_review.json',
    notify_details=True,
)
blocked_config = ProductionHealthCheckConfig(
    name='production_health_blocked_objective_loss',
    registry_path=CARD / 'production_registry.json',
    strategy_id='paper_blocked_objective_loss_strategy',
    output_path=CARD / 'production_health_blocked.json',
    notify_details=True,
)
review_result = run_production_health_check_once(review_config, lock_dir=CARD / 'locks', notify=events.append)
blocked_result = run_production_health_check_once(blocked_config, lock_dir=CARD / 'locks', notify=events.append)

assert review_result.status == 'success'
assert review_result.payload['health_status'] == 'healthy'
assert review_result.payload['paper_execution_objective_loss_status'] == 'review'
assert review_result.payload['paper_execution_objective_loss_top_component'] == 'ranking_loss'
assert blocked_result.status == 'success'
assert blocked_result.payload['health_status'] == 'paper_execution_objective_loss_blocked'
assert blocked_result.payload['is_healthy'] is False
assert blocked_result.payload['paper_execution_objective_loss_status'] == 'block'
assert blocked_result.payload['paper_execution_objective_loss_reward_surface'] == 'active_rl_reward'
events

['production_health_review_objective_loss:success | strategy=paper_review_objective_loss_strategy | health=healthy | missing=0 | dashboard_manifest=analysis_manifest/complete | paper_execution=schedule.paper_execution.v1/success | objective_loss=objective.loss.v1/review top=ranking_loss:1 reward=retired | manifest=/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/production_cases/paper_review_objective_loss_strategy/production_manifest.json',
 'production_health_blocked_objective_loss:success | strategy=paper_blocked_objective_loss_strategy | health=paper_execution_objective_loss_blocked | missing=0 | dashboard_manifest=analysis_manifest/complete | paper_execution=schedule.paper_execution.v1/success | objective_loss=objective.loss.v1/block top=ic_loss:0.9774 reward=active_rl_reward | manifest=/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/production_cases/paper_blocked_objective_loss_strategy/production_manifest.json']

In [6]:
dashboard_payload = {
    'schema_version': 'dashboard.model_matrix.v1',
    'artifact_id': 'production_health_objective_loss_dashboard',
    'manifest_source': {
        'kind': 'analysis_manifest',
        'path': 'matrix_card/analysis_manifest.json',
        'manifest_hash': 'd' * 64,
        'producer': 'src.analysis.exports',
    },
    'top_models': [{'rank': 1, 'model': 'ridge_mse__all_features', 'avg_position': 0.5}],
    'production_health': blocked_result.payload,
}
view_model = build_dashboard_view_model(dashboard_payload)
html = render_dashboard_html(view_model, title='SOXL Production Health Objective Loss')
assert 'Objective Loss' in html
assert 'objective.loss.v1 / block' in html
assert 'ic_loss: 0.98' in html
(CARD / 'dashboard_preview.html').write_text(html, encoding='utf-8')
view_model['production_health']['paper_execution_objective_loss_status']

'block'

In [7]:
decision = build_architecture_extension_decision(
    decision_id='production-health-objective-loss-v1',
    target_surface='src/schedule,src/dashboard',
    change_type='production_health_evidence',
    reason='需要让 production health 从 paper execution evidence 读取 objective.loss.v1 gate，统一展示执行前 review/block 原因。',
    impact_scope=(
        'src/schedule/pipelines.py',
        'src/dashboard/page.py',
        'research/production_health_objective_loss_v1',
    ),
    evidence_notebook='research/production_health_objective_loss_v1/production_health_objective_loss_v1.ipynb',
    verification_commands=(
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests/test_schedule_production_health.py tests/test_dashboard_page_renderer.py -q',
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests -q',
    ),
    alternatives_considered=(
        '只在 paper execution 通知显示 objective loss，不进入 production health',
        '让 dashboard 直接读取 intent_ledger sidecar，违反统一 manifest/export 边界',
    ),
    constraints=(
        'production health 只能读取 production manifest 声明的 artifact',
        'v1 仍只允许 paper/sim，不触发真实 broker 下单',
    ),
    created_at='2026-06-19T00:00:00Z',
)
decision_result = write_architecture_extension_decision(
    output_dir=CARD,
    decision=decision,
    registry_path=CARD / 'architecture_extension_registry.json',
)
decision_result

{'schema_version': 'core.architecture_extension_write_result.v1',
 'decision_path': '/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/production-health-objective-loss-v1.json',
 'registry_path': '/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/architecture_extension_registry.json',
 'review_path': '/Users/rziiiii/dev/workspace/research/production_health_objective_loss_v1/architecture_extension_review.md',
 'decision_hash': '31a714c0db8e60cafb51c1ed9b984753e423aedd0b7cb0672cbf89e833e0eef6',
 'registry_hash': 'a4bf788c8dae1d828495dd991662a20d0336d12b1954a4aa68b2292ce4bd1aeb'}

In [8]:
summary = {
    'schema_version': 'research.production_health_objective_loss.v1',
    'status': 'completed',
    'review_case': {
        'health_status': review_result.payload['health_status'],
        'is_healthy': review_result.payload['is_healthy'],
        'objective_loss_status': review_result.payload['paper_execution_objective_loss_status'],
        'objective_loss_reward_surface': review_result.payload['paper_execution_objective_loss_reward_surface'],
        'objective_loss_top_component': review_result.payload['paper_execution_objective_loss_top_component'],
        'objective_loss_top_value': review_result.payload['paper_execution_objective_loss_top_value'],
    },
    'blocked_case': {
        'health_status': blocked_result.payload['health_status'],
        'is_healthy': blocked_result.payload['is_healthy'],
        'objective_loss_status': blocked_result.payload['paper_execution_objective_loss_status'],
        'objective_loss_reward_surface': blocked_result.payload['paper_execution_objective_loss_reward_surface'],
        'objective_loss_top_component': blocked_result.payload['paper_execution_objective_loss_top_component'],
        'objective_loss_top_value': blocked_result.payload['paper_execution_objective_loss_top_value'],
    },
    'notification_events': events,
    'dashboard_preview': 'research/production_health_objective_loss_v1/dashboard_preview.html',
    'decision_result': decision_result,
    'conclusion': 'production health 已从 paper execution evidence 读取 objective.loss.v1；review 保持 healthy，block 标记为 paper_execution_objective_loss_blocked，并在 dashboard 页面显示。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(CARD / 'notification_events.json').write_text(json.dumps({'events': events}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
summary

{'schema_version': 'research.production_health_objective_loss.v1',
 'status': 'completed',
 'review_case': {'health_status': 'healthy',
  'is_healthy': True,
  'objective_loss_status': 'review',
  'objective_loss_reward_surface': 'retired',
  'objective_loss_top_component': 'ranking_loss',
  'objective_loss_top_value': 1.0},
 'blocked_case': {'health_status': 'paper_execution_objective_loss_blocked',
  'is_healthy': False,
  'objective_loss_status': 'block',
  'objective_loss_reward_surface': 'active_rl_reward',
  'objective_loss_top_component': 'ic_loss',
  'objective_loss_top_value': 0.9774},
 'notification_events': ['production_health_review_objective_loss:success | strategy=paper_review_objective_loss_strategy | health=healthy | missing=0 | dashboard_manifest=analysis_manifest/complete | paper_execution=schedule.paper_execution.v1/success | objective_loss=objective.loss.v1/review top=ranking_loss:1 reward=retired | manifest=/Users/rziiiii/dev/workspace/research/production_health_ob

## 中文结论

- 做了什么：让 `ProductionHealthCheckConfig` 通过 production manifest 声明的 `paper_execution_result` / `intent_ledger` 读取 `objective.loss.v1` gate，并把摘要写入 production health export、通知和 dashboard 页面。
- 核心指标：`objective_loss_status=review` 时 production health 保持 `healthy`；`objective_loss_status=block` 时 health 变为 `paper_execution_objective_loss_blocked` 且 `is_healthy=False`。
- Regime 差异：本实验不做市场 regime 对比，只验证生产健康链路；模型在不同 regime/区间的差异仍由 `src/analysis` 产物负责。
- 是否晋升：晋升为框架能力，不代表任何策略晋升；v1 仍只支持 paper/sim。
- 风险：production health 只读取已写入 paper execution metadata 的摘要；如果上游没有执行 paper execution，仍需通过 missing path 和 paper job status 判断。
- 下一步：把 production notebook 的真实 summary/manifest 同步记录这些 production health objective loss 字段。
